In [1]:
%%capture
# Installs Unsloth and compatible library versions (takes 2-4 minutes; output is hidden).
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas

In [2]:
import psutil, torch, gc
def mem():
    print(f"RAM {psutil.Process().memory_info().rss/1e9:.1f} GB | VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")
mem()

RAM 0.6 GB | VRAM 0.0 GB


In [3]:
import json
from datasets import load_dataset
train_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="train")
test_data  = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")
print(train_data.features)
print(train_data[0]["entities"])

INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."

MAX_SIDE = 1024
def shrink(img):
    img = img.convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    return img

def to_conversation(sample):
    target = json.dumps({k: sample["entities"][k] for k in ["company", "date", "address", "total"]})
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": INSTRUCTION},
                                     {"type": "image", "image": shrink(sample["image"])}]},
        {"role": "assistant", "content": [{"type": "text", "text": target}]},
    ]}

N_TRAIN = 240
train_conversations = [to_conversation(train_data[i]) for i in range(N_TRAIN)]
mem()

README.md:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  319MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

{'image': Image(mode=None, decode=True), 'key': Value('string'), 'image_size': {'width': Value('int64'), 'height': Value('int64')}, 'entities': {'company': Value('string'), 'date': Value('string'), 'address': Value('string'), 'total': Value('string')}, 'words': List(Value('string')), 'bboxes': List(List(Value('int64')))}
{'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD', 'date': '25/12/2018', 'address': 'NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.', 'total': '9.00'}
RAM 1.5 GB | VRAM 0.0 GB


In [6]:
import re, pandas as pd

def parse_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        return None

def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=200):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(image.convert("RGB"), input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FIELDS = ["company", "date", "address", "total"]
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().lower()

def evaluate_vision(ds, n=50):
    rows = []
    for i in range(n):
        s = ds[i]
        raw = generate_from_image(shrink(s["image"]))
        pred = parse_json(raw) or {}
        rows.append({"valid_json": int(bool(pred)),
                     **{f: int(norm(pred.get(f, "")) == norm(s["entities"][f])) for f in FIELDS},
                     "raw": raw})
        print(f"\r{i+1}/{n}", end="")
    df = pd.DataFrame(rows)
    print("\n", df[["valid_json"] + FIELDS].mean().mul(100).round(1).to_string())
    return df

# smoke test on ONE receipt first
print(generate_from_image(shrink(test_data[0]["image"])))
mem()

```json
{
  "company": "OJC MARKETING SDN BHD",
  "date": "15/01/2019",
  "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR",
  "total": "193.00"
}
```
RAM 3.0 GB | VRAM 8.1 GB


In [7]:
gc.collect(); torch.cuda.empty_cache(); mem()

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = True, finetune_language_layers = True,
    finetune_attention_modules = True, finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_conversations,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        max_grad_norm = 0.3, warmup_ratio = 0.03, max_steps = 60,
        learning_rate = 2e-4, logging_steps = 5, optim = "adamw_8bit",
        weight_decay = 0.001, lr_scheduler_type = "cosine", seed = 3407,
        output_dir = "outputs", report_to = "none",
        remove_unused_columns = False, dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True}, max_length = 2048,
    ),
)
mem()
trainer_stats = trainer.train()
print(f"Training finished in {trainer_stats.metrics['train_runtime']/60:.1f} minutes.")
mem()

RAM 3.0 GB | VRAM 8.1 GB


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16
RAM 3.1 GB | VRAM 8.3 GB


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,921,280 of 5,153,099,296 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.159256
10,0.768681
15,0.165716
20,0.092563
25,0.094162
30,0.082685
35,0.072650
40,0.058137
45,0.082968
50,0.086920


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Training finished in 6.1 minutes.
RAM 3.4 GB | VRAM 8.4 GB


In [8]:
  model.save_pretrained("gemma_4_receipts_lora")
   processor.save_pretrained("gemma_4_receipts_lora")
   !zip -qr gemma_4_receipts_lora.zip gemma_4_receipts_lora

Unsloth: Restored added_tokens_decoder metadata in gemma_4_receipts_lora/tokenizer_config.json.


In [9]:
   model.save_pretrained("gemma_4_receipts_lora")
   processor.save_pretrained("gemma_4_receipts_lora")
   !zip -qr gemma_4_receipts_lora.zip gemma_4_receipts_lora

In [10]:
FastVisionModel.for_inference(model)
finetuned_df = evaluate_vision(test_data, n=50)

comparison = pd.DataFrame({
    "Plain (%)": baseline_df[["valid_json"] + FIELDS].mean().mul(100).round(1),
    "Fine-tuned (%)": finetuned_df[["valid_json"] + FIELDS].mean().mul(100).round(1),
})
comparison["Change (points)"] = comparison["Fine-tuned (%)"] - comparison["Plain (%)"]
print(comparison.to_string())

50/50
 valid_json    100.0
company        84.0
date           94.0
address        48.0
total          78.0


NameError: name 'baseline_df' is not defined

In [11]:
def evaluate_vision(ds, n=50, instruction=INSTRUCTION):
    rows = []
    for i in range(n):
        s = ds[i]
        raw = generate_from_image(shrink(s["image"]), instruction=instruction)
        pred = parse_json(raw) or {}
        rows.append({"valid_json": int(bool(pred)),
                     **{f: int(norm(pred.get(f, "")) == norm(s["entities"][f])) for f in FIELDS},
                     "raw": raw})
        print(f"\r{i+1}/{n}", end="")
    df = pd.DataFrame(rows)
    print("\n", df[["valid_json"] + FIELDS].mean().mul(100).round(1).to_string())
    return df

DETAILED = """You are extracting fields from a scanned retail receipt.
Return ONLY a JSON object with exactly these keys: "company", "date", "address", "total".
- company: the business name as printed at the top of the receipt
- date: the transaction date exactly as printed
- address: the full street address as one string
- total: the final amount paid, as printed, without a currency symbol
No explanation and no markdown fences."""

with model.disable_adapter():
    baseline_df = evaluate_vision(test_data, n=50)                       # plain model, short prompt
    prompted_df = evaluate_vision(test_data, n=50, instruction=DETAILED) # plain model, detailed prompt

cols = ["valid_json"] + FIELDS
comparison = pd.DataFrame({
    "Plain (%)":        baseline_df[cols].mean().mul(100).round(1),
    "Plain+prompt (%)": prompted_df[cols].mean().mul(100).round(1),
    "Fine-tuned (%)":   finetuned_df[cols].mean().mul(100).round(1),
})
print(comparison.to_string())

baseline_df.to_csv("baseline.csv"); prompted_df.to_csv("prompted.csv"); finetuned_df.to_csv("finetuned.csv")

50/50
 valid_json    100.0
company        76.0
date           80.0
address        22.0
total          70.0
50/50
 valid_json    100.0
company        74.0
date           78.0
address        28.0
total          68.0
            Plain (%)  Plain+prompt (%)  Fine-tuned (%)
valid_json      100.0             100.0           100.0
company          76.0              74.0            84.0
date             80.0              78.0            94.0
address          22.0              28.0            48.0
total            70.0              68.0            78.0


In [17]:
with model.disable_adapter():
    base_more = evaluate_range(test_data, 50, N_MORE)
baseline150 = pd.concat([baseline_df, base_more])
baseline150.to_csv("baseline150.csv"); mem()

150/150RAM 3.9 GB | VRAM 8.3 GB


In [12]:
   def show_errors(df, ds, field, k=8):
       for i in df.index[df[field] == 0][:k]:
           pred = (parse_json(df.loc[i, "raw"]) or {}).get(field, "")
           print(f"[{i}] gold: {ds[i]['entities'][field]!r}\n     pred: {pred!r}\n")

   show_errors(baseline_df, test_data, "address")
   show_errors(finetuned_df, test_data, "address")

[0] gold: 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR'
     pred: 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR'

[2] gold: 'NO.59 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81750 JOHOR BAHRU'
     pred: 'NO.52 JALAN PERMAS 95 BANDAR SARU PERMAS JAYA 81750 JOHOR BAHRU'

[3] gold: 'KM 458.4 BKT LANJAN UTARA, L/RAYA UTARA SELATAN,SG BULOH 47000 SUNGAI BUL'
     pred: 'KM 458.4 BKT LANJAN UTARA, L/RAYA UTARA SELATAN,SG BULOH'

[4] gold: 'LEVEL 6, BANGUNAN TH, DAMANSARA UPTOWN3 NO.3, JALAN SS21/39,47400 PETALING JAYA SELANGOR'
     pred: 'Level 6, Bangunan TH, Damansara Uptown3 No.3, Jalan SS21/39, 47400 Petaling Jaya Selangor'

[5] gold: 'LOT 13, JALAN IPOH, KG BATU 30, ULU YAM LAMA 44300 BTG KALI, SELANGOR.'
     pred: 'LOT 13, JALAN IPOH, KG BATU 30, ULU YAM LAMA, SELANGOR'

[8] gold: 'NO. 5-7, JALAN MAHAGONI 7/1, SEKYSEN 4, BANDAR UTAMA, 44300 BATANG KALI, SELANGOR.'
     pred: 'NO. 5-7, Jalan Mahagoni 7/1, Sekysen 4, Bandar Utama, 44300 Batang Kali, Sel

In [13]:
   import difflib
   sim = lambda a, b: difflib.SequenceMatcher(None, norm(a), norm(b)).ratio()

   def fuzzy(df, ds, thr=0.9):
       n = len(df)
       return {f: round(sum(sim((parse_json(df.loc[i, "raw"]) or {}).get(f, ""), ds[i]["entities"][f]) >= thr
                            for i in range(n)) / n * 100, 1) for f in FIELDS}

   print({"plain": fuzzy(baseline_df, test_data), "prompted": fuzzy(prompted_df, test_data), "tuned": fuzzy(finetuned_df, test_data)})

{'plain': {'company': 84.0, 'date': 82.0, 'address': 58.0, 'total': 70.0}, 'prompted': {'company': 82.0, 'date': 80.0, 'address': 60.0, 'total': 68.0}, 'tuned': {'company': 88.0, 'date': 98.0, 'address': 80.0, 'total': 80.0}}


In [19]:
comparison = pd.DataFrame({
    "Plain (%)":        baseline150[cols].mean().mul(100).round(1),
    "Plain+prompt (%)": prompted150[cols].mean().mul(100).round(1),
    "Fine-tuned (%)":   finetuned150[cols].mean().mul(100).round(1),
})
comparison["Tuned - Plain"] = comparison["Fine-tuned (%)"] - comparison["Plain (%)"]
print(comparison.to_string())

from scipy.stats import binomtest
for f in FIELDS:
    tuned_only = int(((finetuned150[f] == 1) & (baseline150[f] == 0)).sum())
    plain_only = int(((finetuned150[f] == 0) & (baseline150[f] == 1)).sum())
    n = tuned_only + plain_only
    p = binomtest(tuned_only, n, 0.5).pvalue if n else float("nan")
    print(f"{f:8s} tuned-only right: {tuned_only:3d} | plain-only right: {plain_only:3d} | p = {p:.3f}")

            Plain (%)  Plain+prompt (%)  Fine-tuned (%)  Tuned - Plain
valid_json      100.0             100.0           100.0            0.0
company          76.0              78.7            82.7            6.7
date             88.0              85.3            94.0            6.0
address          27.3              33.3            54.7           27.4
total            70.7              70.7            80.0            9.3
company  tuned-only right:  16 | plain-only right:   6 | p = 0.052
date     tuned-only right:  10 | plain-only right:   1 | p = 0.012
address  tuned-only right:  49 | plain-only right:   8 | p = 0.000
total    tuned-only right:  16 | plain-only right:   2 | p = 0.001


In [15]:
def evaluate_range(ds, start, end, instruction=INSTRUCTION):
    rows = []
    for i in range(start, end):
        s = ds[i]
        raw = generate_from_image(shrink(s["image"]), instruction=instruction)
        pred = parse_json(raw) or {}
        rows.append({"idx": i, "valid_json": int(bool(pred)),
                     **{f: int(norm(pred.get(f, "")) == norm(s["entities"][f])) for f in FIELDS},
                     "raw": raw})
        print(f"\r{i+1}/{end}", end="")
        if i % 25 == 0:
            gc.collect()
    return pd.DataFrame(rows).set_index("idx")

N_MORE = 150
cols = ["valid_json"] + FIELDS

In [16]:
FastVisionModel.for_inference(model)
ft_more = evaluate_range(test_data, 50, N_MORE)
finetuned150 = pd.concat([finetuned_df, ft_more])
finetuned150.to_csv("finetuned150.csv"); mem()

150/150RAM 3.9 GB | VRAM 8.3 GB


In [18]:
with model.disable_adapter():
    prompt_more = evaluate_range(test_data, 50, N_MORE, instruction=DETAILED)
prompted150 = pd.concat([prompted_df, prompt_more])
prompted150.to_csv("prompted150.csv"); mem()

150/150RAM 3.9 GB | VRAM 8.3 GB


In [5]:
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("Model and processor loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Model and processor loaded.


In [20]:
fz = pd.DataFrame({"plain": fuzzy(baseline150, test_data),
                   "prompted": fuzzy(prompted150, test_data),
                   "tuned": fuzzy(finetuned150, test_data)})
print(fz.to_string())

         plain  prompted  tuned
company   82.0      86.0   91.3
date      91.3      88.7   98.0
address   71.3      70.7   87.3
total     72.0      72.0   81.3


In [21]:
def categorize(pred, gold):
    p, g = norm(pred), norm(gold)
    if not p: return "missing"
    if sim(p, g) >= 0.9: return "near-miss (format/punctuation)"
    if g in p or p in g: return "partial (too much or too little)"
    return "wrong text (misread or wrong field)"

rows = []
for i in finetuned150.index:
    for f in FIELDS:
        if finetuned150.loc[i, f] == 0:
            pred = (parse_json(finetuned150.loc[i, "raw"]) or {}).get(f, "")
            rows.append({"field": f, "type": categorize(pred, test_data[i]["entities"][f])})
errs = pd.DataFrame(rows)
print(pd.crosstab(errs.field, errs.type))

type     near-miss (format/punctuation)  partial (too much or too little)  \
field                                                                       
address                              49                                 7   
company                              13                                 3   
date                                  6                                 0   
total                                 2                                 7   

type     wrong text (misread or wrong field)  
field                                         
address                                   12  
company                                   10  
date                                       3  
total                                     21  


In [22]:
def show(field, kind, k=10):
    shown = 0
    for i in finetuned150.index:
        if finetuned150.loc[i, field] == 1: continue
        pred = (parse_json(finetuned150.loc[i, "raw"]) or {}).get(field, "")
        gold = test_data[i]["entities"][field]
        if categorize(pred, gold).startswith(kind):
            print(f"[{i}] gold: {gold!r}\n     pred: {pred!r}\n")
            shown += 1
            if shown == k: break

show("total", "wrong")
show("total", "partial")
show("company", "wrong")

[2] gold: '436.20'
     pred: '438.20'

[3] gold: '4.90'
     pred: '5.00'

[6] gold: '6.35'
     pred: '6.36'

[8] gold: '8.00'
     pred: '10.00'

[10] gold: '169.80'
     pred: '169.78'

[18] gold: '4.20'
     pred: '46.00'

[29] gold: '$6.60'
     pred: '$8.60'

[34] gold: '66.15'
     pred: '66.17'

[41] gold: '5.05'
     pred: '5.04'

[60] gold: '8.40'
     pred: '8 40'

[37] gold: '110.00'
     pred: 'RM 110.00'

[76] gold: '7.42'
     pred: 'RM7.42'

[123] gold: '6.60'
     pred: 'RM 6.60'

[124] gold: '21.50'
     pred: 'RM 21.50'

[125] gold: '6.20'
     pred: 'RM 6.20'

[126] gold: 'RM59.00'
     pred: '59.00'

[129] gold: 'RM42.10'
     pred: '42.10'

[3] gold: 'PETRON BKT LANJAN SB'
     pred: 'ALSERKAM ENTERPRISE'

[10] gold: '99 SPEED MART S/B'
     pred: 'TAMAN BERKELEY'

[13] gold: 'OLD TOWN KOPITAM SND BHD'
     pred: 'OLDTOWN WHITE COFFEE'

[30] gold: 'CONTENTO'
     pred: 'PERMAS'

[40] gold: 'POPULAR BOOK CO. (M) SDN BHD'
     pred: 'CHERRAS LEISURE MALL'

[79] gol

In [23]:
show("total", "wrong")

[2] gold: '436.20'
     pred: '438.20'

[3] gold: '4.90'
     pred: '5.00'

[6] gold: '6.35'
     pred: '6.36'

[8] gold: '8.00'
     pred: '10.00'

[10] gold: '169.80'
     pred: '169.78'

[18] gold: '4.20'
     pred: '46.00'

[29] gold: '$6.60'
     pred: '$8.60'

[34] gold: '66.15'
     pred: '66.17'

[41] gold: '5.05'
     pred: '5.04'

[60] gold: '8.40'
     pred: '8 40'



In [25]:
import os
os.makedirs("demo_receipts", exist_ok=True)
paths = []
for i in [0, 1, 2, 3]:
    p = f"demo_receipts/receipt_{i}.jpg"
    shrink(test_data[i]["image"]).save(p)
    paths.append(p)
    print(i, test_data[i]["entities"])   # gold answers, to compare against during the demo

0 {'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR', 'total': '193.00'}
1 {'company': 'OJC MARKETING SDN BHD', 'date': '02/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR', 'total': '170.00'}
2 {'company': 'PERNIAGAAN ZHENG HUI', 'date': '09/02/2018', 'address': 'NO.59 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81750 JOHOR BAHRU', 'total': '436.20'}
3 {'company': 'PETRON BKT LANJAN SB', 'date': '01/02/2018', 'address': 'KM 458.4 BKT LANJAN UTARA, L/RAYA UTARA SELATAN,SG BULOH 47000 SUNGAI BUL', 'total': '4.90'}


In [26]:
import gradio as gr, json
gr.close_all()
FastVisionModel.for_inference(model)

def pretty(raw):
    parsed = parse_json(raw)
    return json.dumps(parsed, indent=2, ensure_ascii=False) if parsed else raw

def extract(image):
    if image is None:
        raise gr.Error("Upload a receipt image first.")
    img = shrink(image)
    with model.disable_adapter():
        plain = generate_from_image(img)
    tuned = generate_from_image(img)
    return pretty(plain), pretty(tuned)

demo = gr.Interface(
    fn=extract,
    inputs=gr.Image(type="pil", label="Receipt"),
    outputs=[gr.Textbox(label="Plain Gemma 4 E2B", lines=8),
             gr.Textbox(label="Fine-tuned Gemma 4 E2B", lines=8)],
    examples=paths,
    title="Receipt Extraction: plain vs fine-tuned",
    description="Extracts company, date, address and total from a receipt. Fine-tuned on 240 SROIE receipts with LoRA.",
)
demo.queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e812754565680d60e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [28]:
import os
from datasets import load_dataset

test_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")

os.makedirs("demo_receipts", exist_ok=True)
for i in range(6):
    img = test_data[i]["image"].convert("RGB")
    img.thumbnail((1024, 1024))
    img.save(f"demo_receipts/receipt_{i}.jpg")
    print(i, test_data[i]["entities"])   # the correct answers, to check the demo against

0 {'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR', 'total': '193.00'}
1 {'company': 'OJC MARKETING SDN BHD', 'date': '02/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR', 'total': '170.00'}
2 {'company': 'PERNIAGAAN ZHENG HUI', 'date': '09/02/2018', 'address': 'NO.59 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81750 JOHOR BAHRU', 'total': '436.20'}
3 {'company': 'PETRON BKT LANJAN SB', 'date': '01/02/2018', 'address': 'KM 458.4 BKT LANJAN UTARA, L/RAYA UTARA SELATAN,SG BULOH 47000 SUNGAI BUL', 'total': '4.90'}
4 {'company': 'GERBANG ALAF RESTAURANTS SDN BHD', 'date': '18/01/2018', 'address': 'LEVEL 6, BANGUNAN TH, DAMANSARA UPTOWN3 NO.3, JALAN SS21/39,47400 PETALING JAYA SELANGOR', 'total': '38.90'}
5 {'company': 'SIN LIANHAP SDN BHD', 'date': '05/02/2018', 'address': 'LOT 13, JALAN IPOH, KG BATU 30, ULU YAM LAMA 44300 BTG KALI, SELANGOR.', 'total': '7.30'}


In [30]:
import base64
import os
import gradio as gr

gr.close_all()

# Load cooked_logo.png as inline base64 if present in directory
logo_src = ""
if os.path.exists("cooked_logo.png"):
  with open("cooked_logo.png", "rb") as f:
    logo_src = f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# Comprehensive CSS: vector entrance, pulsing dots, animated line bridges, and zero white boxes
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700;800;900&family=JetBrains+Mono:wght@500;700&display=swap');

html { scroll-behavior: smooth; }
body, .gradio-container {
    background-color: #0b0f17 !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    color: #f1f5f9 !important;
}

/* FIX CODEMIRROR & GRADIO INNER WHITE BOXES */
.cm-editor, .cm-scroller, .cm-gutters, .cm-content, .cm-line {
    background-color: #05080f !important;
    color: #38bdf8 !important;
    border-color: #1e293b !important;
}
.cm-gutters {
    border-right: 1px solid #1e293b !important;
    color: #475569 !important;
}
.gradio-container textarea, .gradio-container .cm-editor {
    border: 1px solid #1e293b !important;
    border-radius: 14px !important;
    background: #05080f !important;
}

/* REMOVE WHITE BOXES INSIDE GRADIO UPLOAD CONTAINER */
.gradio-container .image-container,
.gradio-container .upload-container,
.gradio-container .empty,
.gradio-container [data-testid="image"],
.gradio-container .wrap {
    background-color: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

/* ANIMATIONS */
@keyframes vector-enter-left {
  0% { opacity: 0; transform: translateX(-40px) scale(0.92); }
  100% { opacity: 1; transform: translateX(0) scale(1); }
}
@keyframes vector-enter-right {
  0% { opacity: 0; transform: translateX(40px) scale(0.92); }
  100% { opacity: 1; transform: translateX(0) scale(1); }
}
@keyframes line-draw {
  0% { stroke-dashoffset: 100; opacity: 0.2; }
  50% { opacity: 1; }
  100% { stroke-dashoffset: 0; opacity: 1; }
}
@keyframes pulse-ring {
  0% { transform: scale(0.97); box-shadow: 0 0 0 0 rgba(249, 115, 22, 0.45); }
  70% { transform: scale(1); box-shadow: 0 0 0 16px rgba(249, 115, 22, 0); }
  100% { transform: scale(0.97); box-shadow: 0 0 0 0 rgba(249, 115, 22, 0); }
}
@keyframes pulse-dot {
  0%, 100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); }
  50% { transform: scale(1.35); box-shadow: 0 0 10px 3px rgba(16, 185, 129, 0.9); }
}
@keyframes pulse-dot-plain {
  0%, 100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 255, 255, 0.5); }
  50% { transform: scale(1.35); box-shadow: 0 0 8px 3px rgba(255, 255, 255, 0.8); }
}

.anim-vector-left { animation: vector-enter-left 1.2s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
.anim-vector-right { animation: vector-enter-right 1.2s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
.dot-tuned { animation: pulse-dot 2s infinite ease-in-out; }
.dot-plain { animation: pulse-dot-plain 2s infinite ease-in-out; }
.connecting-line { stroke-dasharray: 100; animation: line-draw 2s ease-in-out infinite alternate; }

/* Sticky Navbar */
.cooked-nav {
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: rgba(11, 15, 23, 0.85);
    backdrop-filter: blur(12px);
    border: 1px solid #1e293b;
    border-radius: 20px;
    padding: 12px 28px;
    margin-bottom: 25px;
    position: sticky;
    top: 10px;
    z-index: 100;
}
.nav-links a {
    color: #94a3b8;
    text-decoration: none;
    font-weight: 700;
    font-size: 14px;
    margin-left: 20px;
    transition: color 0.2s;
}
.nav-links a:hover { color: #f97316; }

/* Circular Upload Dropzone */
.circular-upload {
    border: 2px dashed #f97316 !important;
    background: radial-gradient(circle, #172233 0%, #0d1421 100%) !important;
    border-radius: 28px !important;
    padding: 24px !important;
    animation: pulse-ring 3.5s infinite;
}

/* Action Button */
.extract-btn {
    background: linear-gradient(90deg, #f97316, #ea580c) !important;
    color: #0b0f17 !important;
    font-weight: 900 !important;
    font-size: 16px !important;
    border-radius: 14px !important;
    border: none !important;
    padding: 16px !important;
    box-shadow: 0 10px 25px -5px rgba(249, 115, 22, 0.4) !important;
}
"""

def pretty(raw):
    parsed = parse_json(raw)
    return json.dumps(parsed, indent=2, ensure_ascii=False) if parsed else raw
def generate_plain_vision(image):
    try:
        with model.disable_adapter():
            return pretty(generate_from_image(shrink(image)))
    except Exception as e:
        return f"(Plain model output unavailable: {e})"


def on_compare_image(image):
    if image is None:
        raise gr.Error("Please upload or choose a receipt image first.")
    return generate_plain_vision(image), pretty(generate_from_image(shrink(image)))


with gr.Blocks(
    title="cooked.exe - Raw receipts in. Structured JSON out.", css=CUSTOM_CSS
) as demo:

  # 1. FIXED TOP NAVBAR
  logo_html = (
      f'<img src="{logo_src}" style="height:36px; object-fit:contain;" />'
      if logo_src
      else '<span style="font-family:monospace; font-size:22px; font-weight:900; color:white;">cooked<span style="color:#f97316;">.exe</span></span>'
  )
  gr.HTML(f"""
        <div class="cooked-nav">
            <div style="display:flex; align-items:center; gap:12px;">
                {logo_html}
            </div>
            <div class="nav-links">
                <a href="#demo" style="color:#f97316;">Home</a>
                <a href="#demo">Live Extractor</a>
                <a href="#benchmarks">Benchmark</a>
                <a href="#architecture">Architecture</a>
            </div>
        </div>
    """)

  # 2. HERO HEADLINE
  gr.HTML("""
        <div style="text-align:center; padding: 15px 0 25px 0;">
            <div style="display:inline-block; padding: 4px 14px; border-radius:999px; background:rgba(249,115,22,0.12); border:1px solid rgba(249,115,22,0.3); color:#f97316; font-size:12px; font-weight:bold; font-family:'JetBrains Mono'; margin-bottom:12px;">
                GEMMA 4 VISION E2B LoRA
            </div>
            <h1 style="font-size: 3rem; font-weight: 900; line-height: 1.15; margin: 0 0 10px 0;">
                Raw receipts in.<br/>
                <span style="background: linear-gradient(90deg, #fb923c, #f59e0b); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
                    Structured JSON out.
                </span>
            </h1>
            <p style="color: #94a3b8; font-size: 15px; max-width: 520px; margin: 0 auto;">
                Eliminate manual data entry. Extract company, date, address, and total amount in milliseconds.
            </p>
        </div>
    """)

  # 3. HERO INTERACTIVE ARENA: VECTORS + CIRCULAR UPLOAD
  with gr.Row(elem_id="demo"):
    # Left Vector (Entrance animation on page load)
    with gr.Column(scale=1, min_width=220):
      gr.HTML("""
                <div class="anim-vector-left" style="display:flex; flex-direction:column; align-items:center; justify-content:center; height:100%;">
                    <svg style="width:200px; height:200px;" viewBox="0 0 240 240" fill="none">
                        <line x1="20" y1="40" x2="220" y2="40" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <line x1="20" y1="120" x2="220" y2="120" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <line x1="20" y1="200" x2="220" y2="200" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <polygon points="50,60 190,60 170,180 30,180" fill="rgba(30, 41, 59, 0.4)" stroke="#334155" stroke-width="1.5"/>
                        <polygon points="65,75 175,75 160,165 50,165" fill="rgba(15, 23, 42, 0.8)" stroke="#475569" stroke-width="1.5"/>
                        <line x1="75" y1="95" x2="145" y2="95" stroke="#f97316" stroke-width="3" stroke-linecap="round"/>
                        <line x1="73" y1="110" x2="125" y2="110" stroke="#64748b" stroke-width="2" stroke-linecap="round"/>
                        <line x1="71" y1="125" x2="140" y2="125" stroke="#64748b" stroke-width="2" stroke-linecap="round"/>
                        <line x1="69" y1="140" x2="110" y2="140" stroke="#10b981" stroke-width="2.5" stroke-linecap="round"/>
                        <line x1="25" y1="105" x2="205" y2="105" stroke="#f97316" stroke-width="2"/>
                        <circle cx="120" cy="120" r="70" stroke="rgba(249, 115, 22, 0.25)" stroke-width="1.5" stroke-dasharray="8 8"/>
                    </svg>
                    <div style="font-family:'JetBrains Mono'; font-size:11px; color:#94a3b8; margin-top:8px;">
                        <span style="color:#f97316; font-weight:bold;">240</span> Trained Receipts
                    </div>
                </div>
            """)

    # Center Big Upload
    with gr.Column(scale=3):
      image_input = gr.Image(
          type="pil",
          label="Drop image here or click to upload",
          show_label=False,
          sources=["upload", "clipboard"],
          elem_classes=["circular-upload"],
      )

      gr.HTML("""
                <div style="text-align:center; margin-top:8px; margin-bottom:12px; font-size:12px; color:#94a3b8;">
                    Drag or Upload receipt (supports PNG, JPG, PDF up to 10MB)
                </div>
            """)

      extract_btn = gr.Button(
          "Run Side-by-Side Extraction",
          variant="primary",
          elem_classes=["extract-btn"],
      )

    # Right Vector (Entrance animation on page load)
    with gr.Column(scale=1, min_width=220):
      gr.HTML("""
                <div class="anim-vector-right" style="display:flex; flex-direction:column; align-items:center; justify-content:center; height:100%;">
                    <svg style="width:200px; height:200px;" viewBox="0 0 240 240" fill="none">
                        <polygon points="120,40 180,75 180,145 120,180 60,145 60,75" stroke="#334155" stroke-width="1.5" fill="rgba(15, 23, 42, 0.6)"/>
                        <polygon points="120,60 160,85 160,135 120,160 80,135 80,85" stroke="#10b981" stroke-width="1.5" stroke-dasharray="4 4" fill="rgba(16, 185, 129, 0.05)"/>
                        <circle cx="120" cy="40" r="4" fill="#f97316"/>
                        <circle cx="180" cy="75" r="4" fill="#38bdf8"/>
                        <circle cx="180" cy="145" r="4" fill="#10b981"/>
                        <circle cx="120" cy="180" r="4" fill="#f97316"/>
                        <circle cx="60" cy="145" r="4" fill="#38bdf8"/>
                        <circle cx="60" cy="75" r="4" fill="#10b981"/>
                        <line x1="120" y1="40" x2="120" y2="180" stroke="rgba(249, 115, 22, 0.3)" stroke-width="1"/>
                        <line x1="60" y1="75" x2="180" y2="145" stroke="rgba(56, 189, 248, 0.3)" stroke-width="1"/>
                        <line x1="180" y1="75" x2="60" y2="145" stroke="rgba(16, 185, 129, 0.3)" stroke-width="1"/>
                    </svg>
                    <div style="font-family:'JetBrains Mono'; font-size:11px; color:#94a3b8; margin-top:8px;">
                        LoRA Rank <span style="color:#c084fc; font-weight:bold;">16</span> • <span style="color:#34d399; font-weight:bold;">29.9M</span> Params
                    </div>
                </div>
            """)

  # 4. SIDE-BY-SIDE MODEL OUTPUTS (FIXED DARK THEME)
  with gr.Row():
    plain_out = gr.Code(
        value=(
            "// Plain Gemma 4 Vision output will appear here after"
            " extraction..."
        ),
        label="Plain Gemma 4 Vision (Baseline)",
        language="json",
        lines=8,
    )
    tuned_out = gr.Code(
        value=(
            "// cooked.exe Fine-Tuned Gemma output will appear here after"
            " extraction..."
        ),
        label="cooked.exe Fine-Tuned Gemma (LoRA)",
        language="json",
        lines=8,
    )

  extract_btn.click(
      on_compare_image, inputs=image_input, outputs=[plain_out, tuned_out]
  )

  # 5. FIXED BENCHMARKS: CONNECTED HISTOGRAM WITH ANIMATED DOT-BRIDGES
  gr.HTML("""
        <div id="benchmarks" style="padding-top:60px; margin-top:40px; border-top:1px solid #1e293b;">
            <div style="display:flex; justify-content:space-between; align-items:flex-end; margin-bottom:16px;">
                <div>
                    <h2 style="font-size:26px; font-weight:900; color:white; margin:0 0 4px 0;">Accuracy on 150 Test Receipts (Exact Match)</h2>
                    <p style="color:#94a3b8; font-size:13px; margin:0;">Direct evaluation on unseen Malaysian retail receipts (jsdnrs/ICDAR2019-SROIE).</p>
                </div>
                <div style="display:flex; gap:16px; font-size:12px;">
                    <div style="display:flex; align-items:center; gap:6px;">
                        <span style="width:12px; height:12px; background:#64748b; border-radius:3px;"></span>
                        <span style="color:white; font-weight:bold;">Plain (Exact)</span>
                    </div>
                    <div style="display:flex; align-items:center; gap:6px;">
                        <span style="width:12px; height:12px; background:#10b981; border-radius:3px;"></span>
                        <span style="color:#34d399; font-weight:bold;">Fine-Tuned (Exact)</span>
                    </div>
                </div>
            </div>

            <!-- HISTOGRAM CONTAINER WITH CONNECTED DOT BRIDGES -->
            <div style="background:#0e1524; border:1px solid #22324f; border-radius:18px; padding:45px 20px 20px 20px; box-shadow:0 15px 35px rgba(0,0,0,0.4);">

                <div style="display:flex; justify-content:space-around; align-items:flex-end; height:240px; border-bottom:1px solid #334155; position:relative;">

                    <!-- GROUP 1: COMPANY NAME -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-24px; left:0; right:0; width:100%; height:32px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 20 Q 46 6 69 4" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (76%) -->
                            <div style="width:46px; height:182px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">76.0%</span>
                            </div>
                            <!-- Fine-Tuned Bar (82.7%) -->
                            <div style="width:46px; height:198px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">82.7%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Company Name</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+6.7% gain</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 2: DATE EXTRACTION -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-24px; left:0; right:0; width:100%; height:32px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 20 Q 46 8 69 6" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (88%) -->
                            <div style="width:46px; height:211px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">88.0%</span>
                            </div>
                            <!-- Fine-Tuned Bar (94%) -->
                            <div style="width:46px; height:225px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">94.0%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Date Extraction</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+6.0% gain</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 3: ADDRESS DETECTION (STEEP 2X RISE LINE) -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 (Dramatic rise) -->
                            <svg style="position:absolute; top:-70px; left:0; right:0; width:100%; height:80px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 70 C 38 60 52 18 69 6" fill="none" stroke="#10b981" stroke-width="3" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (27.3%) -->
                            <div style="width:46px; height:65px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">27.3%</span>
                            </div>
                            <!-- Fine-Tuned Bar (54.7%) -->
                            <div style="width:46px; height:131px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">54.7%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Address Detection</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399; font-weight:bold;">+27.4% (2x gain)</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 4: TOTAL AMOUNT -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-24px; left:0; right:0; width:100%; height:32px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 22 Q 46 8 69 6" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (70.7%) -->
                            <div style="width:46px; height:169px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">70.7%</span>
                            </div>
                            <!-- Fine-Tuned Bar (80.0%) -->
                            <div style="width:46px; height:192px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">80.0%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Total Amount</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+9.3% gain</span>
                    </div>

                </div>

                <!-- TABLE: EXACT ONLY, PLAIN TEXT IN CLEAN WHITE -->
                <div style="margin-top:24px; overflow-x:auto;">
                    <table style="width:100%; font-size:11px; text-align:left; border-collapse:collapse; font-family:'JetBrains Mono';">
                        <thead>
                            <tr style="color:#94a3b8; border-bottom:1px solid #1e293b;">
                                <th style="padding:10px;">Field</th>
                                <th style="padding:10px; color:white; font-weight:bold;">Plain (Exact)</th>
                                <th style="padding:10px; color:white; font-weight:bold;">Plain + Prompt (Exact)</th>
                                <th style="padding:10px; color:#34d399; font-weight:bold;">Fine-Tuned (Exact)</th>
                                <th style="padding:10px; color:#34d399; font-weight:bold;">Absolute Gain</th>
                            </tr>
                        </thead>
                        <tbody style="color:#cbd5e1;">
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-sans; font-weight:bold; color:#e2e8f0;">valid_json</td>
                                <td style="padding:10px; color:white;">100.0%</td>
                                <td style="padding:10px; color:white;">100.0%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">100.0%</td>
                                <td style="padding:10px; color:#94a3b8;">Baseline Met</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-sans; font-weight:bold; color:#e2e8f0;">company</td>
                                <td style="padding:10px; color:white;">76.0%</td>
                                <td style="padding:10px; color:white;">78.7%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">82.7%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+6.7%</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-sans; font-weight:bold; color:#e2e8f0;">date</td>
                                <td style="padding:10px; color:white;">88.0%</td>
                                <td style="padding:10px; color:white;">85.3%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">94.0%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+6.0%</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b; background:rgba(16,185,129,0.08);">
                                <td style="padding:10px; font-sans; font-weight:bold; color:white;">address</td>
                                <td style="padding:10px; color:white; font-weight:bold;">27.3%</td>
                                <td style="padding:10px; color:white; font-weight:bold;">33.3%</td>
                                <td style="padding:10px; color:#34d399; font-weight:900;">54.7%</td>
                                <td style="padding:10px; color:#34d399; font-weight:900;">+27.4% (2x)</td>
                            </tr>
                            <tr>
                                <td style="padding:10px; font-sans; font-weight:bold; color:#e2e8f0;">total</td>
                                <td style="padding:10px; color:white;">70.7%</td>
                                <td style="padding:10px; color:white;">70.7%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">80.0%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+9.3%</td>
                            </tr>
                        </tbody>
                    </table>
                </div>

            </div>
        </div>
    """)

  # 6. SETUP & ARCHITECTURE SECTION
  gr.HTML("""
        <div id="architecture" style="padding-top:60px; margin-top:40px; border-top:1px solid #1e293b; padding-bottom:50px;">
            <h2 style="font-size:26px; font-weight:900; color:white; margin:0 0 4px 0;">Model Setup & Architecture</h2>
            <p style="color:#94a3b8; font-size:13px; margin:0 0 16px 0;">Hyperparameters recorded from the 6.1-minute fine-tuning run on NVIDIA T4.</p>

            <div style="display:grid; grid-template-columns: repeat(2, 1fr); gap:12px; font-family:'JetBrains Mono'; font-size:12px;">
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Base Model</span>
                    <span style="color:white; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">unsloth/gemma-4-E2B-it (4-bit)</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">LoRA Parameters</span>
                    <span style="color:#c084fc; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">r=16, 29.9M params (0.58%)</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Training Run</span>
                    <span style="color:#fbbf24; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">60 steps, batch=4, 6.1 mins on T4</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Dataset</span>
                    <span style="color:#34d399; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">ICDAR2019-SROIE (240 train / 150 test)</span>
                </div>
            </div>
        </div>
    """)

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_357/1933474240.py:149: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://056a0c1ff2c72136f6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
